# Thematic-analysis runner

Edit the **CONFIG** cell, then run top to bottom — or run only the sections you need.
The heavy lifting lives in `scripts/` (`load`, `metrics`, `embed`, `embed_analysis`,
`report`); this notebook just wires them together so the analysis is reusable across
datasets.

- **Sections 1–2** (load + ratings) need **no API key**.
- **Sections 3–5** (embeddings) need `OPENAI_API_KEY` in a `.env` at the project root.
- Embeddings are cached to `analysis/cache/`, so re-running is free.

## CONFIG — edit for a new analysis

In [9]:
from pathlib import Path
import sys

# Locate the project root (folder containing scripts/) and make it importable.
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents] if (p / "scripts").is_dir()), _here)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

# ===================== EDIT THIS BLOCK ============================
RUN_NAME = "20comments_v1"                        # names the output subfolder
DATA_FILE = ROOT / "data" / "comments_20.md"      # [Dn] comment corpus

RATING_FILES = {                                  # condition -> rating file
    "nodata":     ROOT / "outputs/ratings/themes/chatgpt5.5_nodata_v1_20_run1.themes-ratings.json",
    "loweffort":  ROOT / "outputs/ratings/themes/chatgpt5.5_loweffort_v1_20_run1.themes-ratings.json",
    "engineered": ROOT / "outputs/ratings/themes/chatgpt5.5_engineered_v2_20_run1.themes-ratings.json",
    "human":      ROOT / "outputs/ratings/themes/human_teddy_v1_20_run1.themes-ratings.json",
}

RESEARCH_QUESTION = (
    "What are the experiences and perspectives of the ALS community as expressed "
    "in their public comments to the FDA regarding ALS drug development policy?"
)
EMBEDDING_MODEL = "text-embedding-3-large"
# =================================================================

OUTPUT_DIR = ROOT / "analysis" / "results" / RUN_NAME
CACHE_PATH = ROOT / "analysis" / "cache" / "embeddings.json"
print("ROOT       :", ROOT)
print("OUTPUT_DIR :", OUTPUT_DIR)

ROOT       : c:\Users\teddy\Downloads\SPUR\ai_thematic_analysis
OUTPUT_DIR : c:\Users\teddy\Downloads\SPUR\ai_thematic_analysis\analysis\results\20comments_v1


## 1. Load  *(no API key)*

In [10]:
import importlib
import load, embed, metrics, embed_analysis as ea, report
for _m in (load, embed, metrics, ea, report):   # pick up edits without restarting the kernel
    importlib.reload(_m)

comments = load.load_comments(DATA_FILE)
sets = load.load_all(RATING_FILES)
print(f"{len(comments)} comments")
for cond, ts in sets.items():
    print(f"  {cond:11s} {len(ts.rated_themes):2d} rated themes "
          f"({sum(len(t.quotes) for t in ts.rated_themes)} quotes)")

20 comments
  nodata      20 rated themes (0 quotes)
  loweffort   15 rated themes (47 quotes)
  engineered   6 rated themes (22 quotes)
  human        9 rated themes (192 quotes)


## 2. Ratings  *(no API key)*

In [11]:
rating_tbl = metrics.rating_table(sets)
theme_ratings = metrics.theme_rating_table(sets)
rating_tbl.round(2)

,n_themes,grounding,researchQuestionFit,interpretationLevel,aiPriorNovelty,analyticalNovelty,n_similarity_pairs,mean_similarity,mean_quotes_per_theme
condition,,,,,,,,,
nodata,20,NaN,4.75,2.35,NaN,2.20,10,3.10,0.00
loweffort,15,4.93,4.93,1.47,1.87,2.27,7,2.43,3.13
engineered,6,5.00,5.00,2.67,2.00,3.17,0,NaN,3.67
human,9,4.78,4.44,3.11,3.88,3.33,0,NaN,21.33


## 3. Embeddings  *(needs `OPENAI_API_KEY` in `.env`)*

In [12]:
from embed import Embedder

embedder = Embedder(model=EMBEDDING_MODEL, cache_path=CACHE_PATH)
texts = ea.all_texts(sets, comments, RESEARCH_QUESTION)
print("cost estimate (unique / cached / to_embed):", embedder.cost_estimate(texts))

vecs = ea.build_vectors(sets, comments, RESEARCH_QUESTION, embedder)
print("vectors ready:", len(vecs))

cost estimate (unique / cached / to_embed): {'unique': 284, 'cached': 284, 'to_embed': 0}
vectors ready: 284


## 4. Embedding analyses

In [13]:
rq_df        = ea.theme_rq_cosine(sets, vecs, RESEARCH_QUESTION)
quote_df     = ea.quote_level_table(sets, vecs, comments)
core_supp    = ea.core_vs_supporting(quote_df)
interp_level = ea.interp_level_cosine(quote_df)
theme_cited  = ea.theme_cited_cosine(sets, vecs, comments)
pairs_df     = ea.theme_pairs(sets, vecs)
sim_val      = ea.similarity_validation(pairs_df)

# Interpretation-level correlation uses only verbatim quotes (have a real source),
# so paraphrased low-effort / quote-free no-data themes do not confound it.
verbatim = quote_df[quote_df["source"].notna()]
interp_corr = ea.interp_level_correlation(verbatim)

print("interp level vs quote-distance correlation:", {k: round(v, 3) for k, v in interp_corr.items()})
sim_val.round(3)

interp level vs quote-distance correlation: {'spearman': 0.201, 'pearson': 0.138, 'n': 214.0}


,n_pairs,n_rated,spearman,pearson,mean_cos_rated,mean_cos_unrated,max_cos_unrated
condition,,,,,,,
engineered,15,0,NaN,NaN,NaN,0.631,0.746
human,36,0,NaN,NaN,NaN,0.632,0.747
loweffort,105,7,-0.089,0.015,0.612,0.515,0.706
nodata,190,10,0.770,0.661,0.624,0.513,0.717


## 5. Reports — write CSVs + `summary.md` + `detailed.md`

In [14]:
from IPython.display import Markdown, display

tables = dict(
    rating_tbl=rating_tbl, theme_ratings=theme_ratings, rq_df=rq_df, quote_df=quote_df,
    core_supp=core_supp, interp_level=interp_level, interp_corr=interp_corr,
    theme_cited=theme_cited, pairs_df=pairs_df, sim_val=sim_val,
)
paths = report.write_reports(OUTPUT_DIR, RUN_NAME, tables)
print('reports written:')
for _p in paths['reports']:
    print('  ', _p)
print('csvs:', len(paths['csvs']))

display(Markdown((OUTPUT_DIR / 'summary.md').read_text(encoding='utf-8')))

reports written:
   c:\Users\teddy\Downloads\SPUR\ai_thematic_analysis\analysis\results\20comments_v1\summary.md
   c:\Users\teddy\Downloads\SPUR\ai_thematic_analysis\analysis\results\20comments_v1\detailed.md
csvs: 9


# Thematic-analysis summary — 20comments_v1

## Ratings by condition
Means over rated themes (non-null values per dimension). Scales are 1–5; for novelty/interpretation, higher = more novel / more interpretive.

| condition | n_themes | grounding | researchQuestionFit | interpretationLevel | aiPriorNovelty | analyticalNovelty | n_similarity_pairs | mean_similarity | mean_quotes_per_theme |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| nodata | 20 | — | 4.75 | 2.35 | — | 2.20 | 10 | 3.10 | 0.00 |
| loweffort | 15 | 4.93 | 4.93 | 1.47 | 1.87 | 2.27 | 7 | 2.43 | 3.13 |
| engineered | 6 | 5.00 | 5.00 | 2.67 | 2.00 | 3.17 | 0 | — | 3.67 |
| human | 9 | 4.78 | 4.44 | 3.11 | 3.88 | 3.33 | 0 | — | 21.33 |

## Theme ↔ research question (cosine)
How close each condition's themes sit to the RQ, on average.

| condition | mean_cos_namedef_rq | mean_cos_nameonly_rq |
| --- | --- | --- |
| engineered | 0.584 | 0.478 |
| human | 0.597 | 0.429 |
| loweffort | 0.513 | 0.341 |
| nodata | 0.533 | 0.368 |

## Core vs supporting quotes (cosine to theme)
`core_minus_supporting` > 0 means core quotes sit closer to the theme.

| condition | core_mean | supporting_mean | n_core | n_supporting | core_minus_supporting |
| --- | --- | --- | --- | --- | --- |
| engineered | 0.597 | 0.517 | 11.000 | 11.000 | 0.080 |
| human | 0.325 | 0.352 | 122.000 | 70.000 | -0.027 |
| loweffort | — | 0.421 | — | 47.000 | — |

## Interpretation level ↔ quote distance
Correlation of a theme's interpretation level with its quote→theme cosine (n=214): Spearman **0.201**, Pearson **0.138**. Negative ⇒ more interpretive themes have looser (less similar) quotes.

## Human similarity vs embedding cosine
Does embedding cosine reproduce the 1–5 theme-similarity ratings, and are unrated (implicitly independent) pairs actually lower-cosine than rated ones?

| condition | n_pairs | n_rated | spearman | pearson | mean_cos_rated | mean_cos_unrated | max_cos_unrated |
| --- | --- | --- | --- | --- | --- | --- | --- |
| engineered | 15 | 0 | — | — | — | 0.631 | 0.746 |
| human | 36 | 0 | — | — | — | 0.632 | 0.747 |
| loweffort | 105 | 7 | -0.089 | 0.015 | 0.612 | 0.515 | 0.706 |
| nodata | 190 | 10 | 0.770 | 0.661 | 0.624 | 0.513 | 0.717 |

## Caveats
- **no-data**: themes are model priors with no data, so there are no quotes; `grounding` and `aiPriorNovelty` are not applicable (shown as `—`).
- **low-effort**: quotes are model paraphrases without verbatim source ids, so quote-to-source provenance is unavailable; the condition instead cites comments inside each definition (`Representative comments: D..`), which drive its theme-to-cited-comment grounding signal.
- **human**: the analysis is hierarchical; the 9 rated subthemes are the unit of analysis and the 2 parent/container nodes (all-null ratings) are excluded.
- Theme-similarity ratings exist only where pairs crossed the rating threshold (no-data, low-effort). Unrated pairs are treated as implicitly independent and checked against their embedding cosine.
